# Phase 2 (repaired) — complete the results for the manuscript

Three faults in the previous version are fixed here.

| Fault | Effect | Fix |
|---|---|---|
| Section 1 deleted every `P2_*` checkpoint | the sweep restarted from epoch 1 each session | the inventory cell no longer deletes anything |
| The cell under "attack every defended model" contained a **duplicate sweep** | sections 5 and 6 trained more models instead of attacking them | replaced with the real attack cell |
| No table/figure cell existed | `table3_tradeoff.txt` could never be regenerated | added |

Also fixed: the shuffled-label control now uses its own checkpoint tag (it
previously overwrote `P2_lam1`), and `train_defended` no longer raises
`UnboundLocalError` when a lambda is already complete.

**Run order after a fresh session:** sections 0 and 1 (no training), then 2, 3,
4, 5, 6. Section 7 is optional Phase 1 housekeeping.

## 0 · Setup — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0 · Setup — paths and constants

In [ ]:
import os, sys, re, json, time, math, pickle, random
import numpy as np

SEED = 1
random.seed(SEED); np.random.seed(SEED)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/PhD_Q1'
    SCRATCH = '/content/scratch'
elif IN_KAGGLE:
    PROJECT = '/kaggle/working/PhD_Q1'
    SCRATCH = '/kaggle/working/scratch'
else:
    PROJECT = os.path.abspath('./PhD_Q1')
    SCRATCH = os.path.abspath('./scratch')

DATA_DIR = os.path.join(PROJECT, 'data')
CKPT_DIR = os.path.join(PROJECT, 'checkpoints')
RES_DIR  = os.path.join(PROJECT, 'results')
for d in (DATA_DIR, CKPT_DIR, RES_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)

PKL_PATH  = os.path.join(DATA_DIR, 'ntu60_3danno.pkl')
CACHE_X   = os.path.join(DATA_DIR, 'ntu60_T64_x.npy')
CACHE_M   = os.path.join(DATA_DIR, 'ntu60_T64_meta.npz')

# geometry / tensor shapes
NUM_NODE, MAX_PERSON, T_LEN, IN_CH = 25, 2, 64, 3

print('project :', PROJECT)
print('pkl     :', PKL_PATH, '->', os.path.exists(PKL_PATH))

## 0 · Setup — torch and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime > Change runtime type > GPU.')

## 1 · Load the NTU annotations

In [ ]:
t0 = time.time()
with open(PKL_PATH, 'rb') as f:
    D = pickle.load(f)
print('loaded in %.1f s' % (time.time() - t0))

anns  = D['annotations']
names = [a['frame_dir'] for a in anns]

setup   = np.array([int(n[1:4])   for n in names], dtype=np.int16)
camera  = np.array([int(n[5:8])   for n in names], dtype=np.int16)
subject = np.array([int(n[9:12])  for n in names], dtype=np.int16)
action  = np.array([a['label']    for a in anns],  dtype=np.int16)

print('samples  :', len(anns))
print('subjects :', len(np.unique(subject)))
print('cameras  :', np.unique(camera))
print('actions  :', len(np.unique(action)))
print('example  :', names[0])
print('keys     :', list(anns[0].keys()))
print('keypoint :', np.asarray(anns[0]['keypoint']).shape, ' (M, T, V, C)')

assert len(np.unique(subject)) == 40, 'expected 40 subjects'
print('\nOK — identity labels present. Proceed.')

## 2 · Preprocessing functions

In [ ]:
def _unit(v):
    n = np.linalg.norm(v)
    return v if n < 1e-8 else v / n

def _rotation_matrix(axis, theta):
    axis = _unit(np.asarray(axis, dtype=np.float64))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([[aa+bb-cc-dd, 2*(bc+ad),   2*(bd-ac)],
                     [2*(bc-ad),   aa+cc-bb-dd, 2*(cd+ab)],
                     [2*(bd+ac),   2*(cd-ab),   aa+dd-bb-cc]], dtype=np.float32)

def _angle_between(v1, v2):
    return np.arccos(np.clip(np.dot(_unit(v1), _unit(v2)), -1.0, 1.0))

def _align(skel, ja, jb, target):
    # rotate the whole sequence so that joint ja -> jb points along `target`
    vec = skel[0, 0, jb] - skel[0, 0, ja]
    if np.linalg.norm(vec) < 1e-8:
        return skel
    axis, angle = np.cross(vec, target), _angle_between(vec, target)
    if np.linalg.norm(axis) < 1e-8 or angle < 1e-8:
        return skel
    return (skel @ _rotation_matrix(axis, angle).T).astype(np.float32)

def pre_normalize(kp):
    kp = np.asarray(kp, dtype=np.float32).copy()
    valid = kp.sum(axis=(0, 2, 3)) != 0
    if valid.any():
        kp = kp[:, valid]
    if kp.shape[1] == 0:
        return np.zeros((1, 1, NUM_NODE, 3), dtype=np.float32)
    mask = kp.sum(axis=-1) != 0
    kp[mask] = kp[mask] - kp[0, 0, 1].copy()          # centre on spine
    kp = _align(kp, 0, 1, np.array([0, 0, 1], np.float32))   # spine     -> z
    kp = _align(kp, 8, 4, np.array([1, 0, 0], np.float32))   # shoulders -> x
    return kp

def resample(kp, t_len=T_LEN):
    M, T, V, Ch = kp.shape
    out = kp[:, np.linspace(0, T - 1, t_len).round().astype(np.int64)]
    if M < MAX_PERSON:
        out = np.concatenate(
            [out, np.zeros((MAX_PERSON - M, t_len, V, Ch), out.dtype)], axis=0)
    return out[:MAX_PERSON]

# quick self-test on synthetic data
_t = np.random.RandomState(0).normal(size=(1, 90, 25, 3)).astype(np.float32)
assert resample(pre_normalize(_t)).shape == (MAX_PERSON, T_LEN, NUM_NODE, 3)
print('preprocessing OK')

## 2 · Build or load the cache

In [ ]:
if os.path.exists(CACHE_X) and os.path.exists(CACHE_M):
    print('cache found, skipping preprocessing')
else:
    N = len(anns)
    X = np.zeros((N, MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    t0 = time.time()
    for i, a in enumerate(anns):
        X[i] = resample(pre_normalize(np.asarray(a['keypoint']))).astype(np.float16)
        if (i + 1) % 5000 == 0:
            el = time.time() - t0
            print('  %6d / %d   %.0f s elapsed, ~%.0f s left'
                  % (i + 1, N, el, el / (i + 1) * (N - i - 1)))
    np.save(CACHE_X, X)
    np.savez(CACHE_M, setup=setup, camera=camera, subject=subject,
             action=action, names=np.array(names))
    print('cached in %.0f s -> %.2f GB' % (time.time() - t0, X.nbytes / 1e9))
    del X

## 2 · Copy the cache to local disk

In [ ]:
# copy to local disk first: Drive I/O is slow for repeated reads
LOCAL_X = os.path.join(SCRATCH, 'x.npy')
if not os.path.exists(LOCAL_X):
    import shutil; shutil.copy(CACHE_X, LOCAL_X)

X = np.load(LOCAL_X)                       # (N, 2, 64, 25, 3) float16
_m = np.load(CACHE_M, allow_pickle=True)
setup, camera, subject, action = _m['setup'], _m['camera'], _m['subject'], _m['action']
names = list(_m['names'])

SUBJ_IDS   = sorted(np.unique(subject).tolist())
SUBJ_LABEL = np.array([SUBJ_IDS.index(s) for s in subject], dtype=np.int64)  # 0..39
ACT_LABEL  = action.astype(np.int64)                                         # 0..59

print('X', X.shape, X.dtype, '%.2f GB' % (X.nbytes / 1e9))
print('action classes', ACT_LABEL.max() + 1, '| identity classes', SUBJ_LABEL.max() + 1)

## 3 · Evaluation protocols (splits)

In [ ]:
XSUB_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38]

SPLITS = {}
SPLITS['xsub'] = (np.where(np.isin(subject, XSUB_TRAIN))[0],
                  np.where(~np.isin(subject, XSUB_TRAIN))[0])

rs, tr, te = np.random.RandomState(SEED), [], []
for s in SUBJ_IDS:                                   # stratified 80/20 per person
    idx = np.where(subject == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr += idx[:k].tolist(); te += idx[k:].tolist()
SPLITS['reid_rand'] = (np.array(sorted(tr)), np.array(sorted(te)))

SPLITS['reid_xview'] = (np.where(np.isin(camera, [1, 2]))[0],
                        np.where(camera == 3)[0])

for k, (a, b) in SPLITS.items():
    print('%-11s train %6d  test %6d  | subjects %2d / %2d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b]))))

assert len(set(subject[SPLITS['xsub'][0]]) & set(subject[SPLITS['xsub'][1]])) == 0
assert len(set(subject[SPLITS['reid_xview'][0]]) &
           set(subject[SPLITS['reid_xview'][1]])) == 40

## 4 · ST-GCN definition

In [ ]:
_INWARD_1B = [(1,2),(2,21),(3,21),(4,3),(5,21),(6,5),(7,6),(8,7),(9,21),
              (10,9),(11,10),(12,11),(13,1),(14,13),(15,14),(16,15),
              (17,1),(18,17),(19,18),(20,19),(22,23),(23,8),(24,25),(25,12)]
INWARD    = [(i-1, j-1) for i, j in _INWARD_1B]
OUTWARD   = [(j, i) for i, j in INWARD]
SELF_LINK = [(i, i) for i in range(NUM_NODE)]

def _edge2mat(link, n):
    A = np.zeros((n, n), dtype=np.float32)
    for i, j in link:
        A[j, i] = 1.0
    return A

def _norm_digraph(A):
    Dl, Dn = A.sum(0), np.zeros_like(A)
    for i in range(A.shape[0]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** -1.0
    return A @ Dn

def build_adjacency():
    return np.stack([_edge2mat(SELF_LINK, NUM_NODE),
                     _norm_digraph(_edge2mat(INWARD, NUM_NODE)),
                     _norm_digraph(_edge2mat(OUTWARD, NUM_NODE))]).astype(np.float32)


class SpatialGC(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.K = A.shape[0]
        self.conv = nn.Conv2d(in_c, out_c * self.K, 1)
        self.register_buffer('A', torch.from_numpy(A.copy()))

    def forward(self, x):
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('nkctv,kvw->nctw', x, self.A.to(x.dtype)).contiguous()


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, residual=True):
        super().__init__()
        self.gcn = SpatialGC(in_c, out_c, A)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(0.1)
        if not residual:
            self.res = None
        elif in_c == out_c and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)),
                                     nn.BatchNorm2d(out_c))

    def forward(self, x):
        r = 0 if self.res is None else self.res(x)
        y = F.relu(self.bn1(self.gcn(x)))
        y = self.drop(self.bn2(self.tcn(y)))
        return F.relu(y + r)


class STGCN(nn.Module):
    def __init__(self, num_class, in_channels=IN_CH, num_person=MAX_PERSON, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * NUM_NODE)
        cfg = [(in_channels, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True),
               (64, 64, 1, True), (64, 128, 2, True), (128, 128, 1, True),
               (128, 128, 1, True), (128, 256, 2, True), (256, 256, 1, True),
               (256, 256, 1, True)]
        self.layers = nn.ModuleList([STGCNBlock(i, o, A, s, r) for i, o, s, r in cfg])
        self.drop, self.fc, self.feat_dim = nn.Dropout(dropout), nn.Linear(256, num_class), 256

    def extract(self, x):                      # x: N, C, T, V, M
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        for l in self.layers:
            x = l(x)
        return F.adaptive_avg_pool2d(x, 1).view(N, M, -1).mean(1)

    def forward(self, x, return_feat=False):
        f = self.extract(x)
        out = self.fc(self.drop(f))
        return (out, f) if return_feat else out


_m = STGCN(60)
_o, _f = _m(torch.randn(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON), return_feat=True)
print('logits', tuple(_o.shape), '| features', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o, _f

## 5 · Training engine

In [ ]:
def _euler(theta):
    cx, cy, cz = np.cos(theta); sx, sy, sz = np.sin(theta)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def augment(x, rot=0.3, scale=0.1):
    R = _euler(np.random.uniform(-rot, rot, 3))
    s = 1.0 + np.random.uniform(-scale, scale)
    mask = (x.sum(-1) != 0)[..., None]
    return np.where(mask, (x @ R.T) * s, 0.0).astype(np.float32)


class SkeletonDS(torch.utils.data.Dataset):
    def __init__(self, X, y, idx, train=False):
        self.X, self.y, self.idx, self.train = X, y, idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j].astype(np.float32)
        if self.train:
            x = augment(x)
        x = np.ascontiguousarray(np.transpose(x, (3, 1, 2, 0)))   # C,T,V,M
        return torch.from_numpy(x), int(self.y[j])


def _load_ckpt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(p, map_location='cpu')


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.numel()
    return 100.0 * correct / total


def run_experiment(tag, labels, split, num_classes, epochs=60, lr=0.1, bs=64,
                   wd=5e-4, warmup=5, workers=2, init_from=None, freeze=False):
    tr_idx, te_idx = SPLITS[split]
    tl = torch.utils.data.DataLoader(SkeletonDS(X, labels, tr_idx, True), bs, shuffle=True,
                                     num_workers=workers, pin_memory=(DEVICE == 'cuda'),
                                     drop_last=True, persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(SkeletonDS(X, labels, te_idx, False), bs * 2,
                                     shuffle=False, num_workers=workers,
                                     pin_memory=(DEVICE == 'cuda'))

    model = STGCN(num_classes).to(DEVICE)
    if init_from:
        sd = _load_ckpt(init_from)['model']
        sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
        print('  init from', os.path.basename(init_from),
              model.load_state_dict(sd, strict=False).missing_keys[:2], '...')
    if freeze:
        for n_, p_ in model.named_parameters():
            if not n_.startswith('fc.'):
                p_.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] split=%s  classes=%d  train=%d  test=%d'
          % (tag, split, num_classes, len(tr_idx), len(te_idx)))
    for ep in range(start, epochs):
        cur_lr = set_lr(ep); model.train(); t0 = time.time(); run = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        acc = evaluate(model, vl)
        hist.append({'epoch': ep, 'loss': run / max(1, len(tl)), 'acc': acc, 'lr': cur_lr})
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'best': max(best, acc), 'hist': hist}, last)
        if acc > best:
            best = acc; torch.save({'model': model.state_dict(), 'acc': acc}, best_p)
        print('  ep %3d/%d  loss %.3f  acc %6.2f%%  best %6.2f%%  lr %.4f  (%.0fs)'
              % (ep + 1, epochs, hist[-1]['loss'], acc, best, cur_lr, time.time() - t0))

    json.dump(hist, open(os.path.join(RES_DIR, tag + '_history.json'), 'w'), indent=1)
    print('[%s] BEST %.2f%%' % (tag, best))
    return {'tag': tag, 'split': split, 'classes': num_classes,
            'best_acc': best, 'ckpt': best_p}

RESULTS = {}
RES_JSON = os.path.join(RES_DIR, 'phase1_results.json')
if os.path.exists(RES_JSON):
    RESULTS = json.load(open(RES_JSON))
    print('loaded previous results:', list(RESULTS))

def save_results():
    json.dump(RESULTS, open(RES_JSON, 'w'), indent=2)

## 1 · Load Phase 1 and inventory Phase 2 — nothing is deleted

This replaces the old cell that cleared `P2_*`. Completed lambdas are detected
and skipped later, so finished training is never repeated.

In [ ]:
import json, os, glob
RESULTS = json.load(open(os.path.join(RES_DIR, 'phase1_results.json')))
print('--- Phase 1 (final, never retrained) ---')
for k, v in RESULTS.items():
    print('  %-24s %.2f%%' % (k, v['best_acc']))

P2_DIR = os.path.join(RES_DIR, 'phase2')
os.makedirs(P2_DIR, exist_ok=True)
SUMMARY = os.path.join(P2_DIR, 'p2_summary.json')
P2 = json.load(open(SUMMARY)) if os.path.exists(SUMMARY) else {}

print('\n--- Phase 2 checkpoints on disk ---')
for f in sorted(glob.glob(os.path.join(CKPT_DIR, 'P2_*_best.pt'))):
    print('  ', os.path.basename(f))

print('\n--- Phase 2 summary entries ---')
for k in sorted(P2, key=lambda s: (not s[0].isdigit(), s)):
    r = P2[k]
    ok = os.path.exists(r.get('ckpt', ''))
    print('  %-10s act %6.2f%%  adv %6.2f%%  ckpt %s'
          % (k, r.get('action_acc', 0), r.get('adv_final', -1),
             'OK' if ok else 'MISSING'))

# the stale table from the broken-adversary run must not be reused
stale = os.path.join(P2_DIR, 'table3_tradeoff.txt')
if os.path.exists(stale):
    os.rename(stale, stale + '.OLD_BROKEN_RUN')
    print('\nrenamed stale table3_tradeoff.txt -> .OLD_BROKEN_RUN')

## 2 · P0 — defended model and training loop (defines only)

In [ ]:
import os, json, math, time, glob
import numpy as np
import torch
import torch.nn as nn

P2_DIR = os.path.join(RES_DIR, 'phase2')
os.makedirs(P2_DIR, exist_ok=True)


class GradReverse(torch.autograd.Function):
    """Identity forwards; multiplies the gradient by -lambda backwards."""
    @staticmethod
    def forward(ctx, x, lamb):
        ctx.lamb = lamb
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lamb * g, None


def grad_reverse(x, lamb):
    return GradReverse.apply(x, lamb)


class DefendedSTGCN(nn.Module):
    """Shared ST-GCN trunk, an action head, and an adversarial identity head."""

    def __init__(self, n_action, n_identity, feat_dim=None):
        super().__init__()
        self.trunk = STGCN(n_action)
        if feat_dim is None:
            feat_dim = self._infer_dim()
        self.feat_dim = feat_dim
        self.act_head = nn.Linear(feat_dim, n_action)
        # FIX 1: input normalisation + capacity. The previous 256-unit head
        # stalled at 16% against a measured linear ceiling of ~30%.
        self.adv_head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, n_identity))

    @torch.no_grad()
    def _infer_dim(self):
        was = self.trunk.training
        self.trunk.eval()
        f = self.trunk.extract(torch.zeros(2, 3, 64, 25, 2))
        if f.ndim > 2:
            f = f.mean(dim=tuple(range(2, f.ndim)))
        self.trunk.train(was)
        return f.shape[1]

    def features(self, x):
        f = self.trunk.extract(x)
        if f.ndim > 2:
            f = f.mean(dim=tuple(range(2, f.ndim)))
        return f

    def forward(self, x, lamb=0.0):
        f = self.features(x)
        return self.act_head(f), self.adv_head(grad_reverse(f, lamb))


class DualDS(torch.utils.data.Dataset):
    """Same clips as SkeletonDS, but returns (x, action_label, identity_label)."""

    def __init__(self, X, a_lab, i_lab, idx, train=False):
        self.base = SkeletonDS(X, a_lab, idx, train)
        self.i_lab, self.idx = i_lab, idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x, a = self.base[i]
        return x, a, int(self.i_lab[self.idx[i]])


def make_identity_map(split='xsub'):
    """Remap the subjects in the split's TRAIN half to 0..K-1; others get -1."""
    tr_idx, _ = SPLITS[split]
    subs = sorted(set(np.asarray(subject)[tr_idx].tolist()))
    lut = {s: k for k, s in enumerate(subs)}
    lab = np.full(len(subject), -1, dtype=np.int64)
    for s, k in lut.items():
        lab[np.asarray(subject) == s] = k
    return lab, len(subs)


ID_LABEL_XSUB, N_ID_XSUB = make_identity_map('xsub')
print('adversarial head covers %d identities (chance %.2f%%)'
      % (N_ID_XSUB, 100.0 / N_ID_XSUB))


def export_trunk(defended_ckpt, out_path):
    """FIX 4: strip the 'trunk.' prefix so run_experiment can load the weights."""
    sd = _load_ckpt(defended_ckpt)['model']
    plain = {k[len('trunk.'):]: v for k, v in sd.items() if k.startswith('trunk.')}
    torch.save({'model': plain}, out_path)
    return out_path


def train_defended(lamb, epochs=60, lr=0.1, bs=64, wd=5e-4, warmup=5,
                   ramp=10, workers=4, adv_lr=1e-3, split='xsub', tag=None):
    """Train one defended model at a given lambda. Resumes automatically."""
    # FIX: an explicit tag lets the shuffled-label control use its own
    # checkpoint path instead of overwriting the lambda = 1 model.
    if tag is None:
        tag = 'P2_lam%s' % ('%g' % lamb).replace('.', 'p')
    tr_idx, te_idx = SPLITS[split]

    tl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, tr_idx, True), bs, shuffle=True,
        num_workers=workers, pin_memory=(DEVICE == 'cuda'), drop_last=True,
        persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, te_idx, False), bs * 2,
        shuffle=False, num_workers=workers, pin_memory=(DEVICE == 'cuda'))

    # FIX 2: the adversary must be scored on clips whose identity label is valid.
    # The xsub TEST half contains only Group Q (label -1), which is why the old
    # 'adv' column read 0.00%. We use a fixed 10% sample of the training half.
    rng = np.random.RandomState(0)
    valid = np.asarray([i for i in tr_idx if ID_LABEL_XSUB[i] >= 0])
    adv_idx = rng.choice(valid, size=min(4000, len(valid)), replace=False)
    avl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, np.sort(adv_idx), False), bs * 2,
        shuffle=False, num_workers=workers, pin_memory=(DEVICE == 'cuda'))

    model = DefendedSTGCN(60, N_ID_XSUB).to(DEVICE)

    # FIX 3: two optimisers. The trunk keeps SGD + cosine; the adversary head
    # gets Adam at a constant rate so it can actually reach its ceiling.
    trunk_params = [p for n_, p in model.named_parameters()
                    if not n_.startswith('adv_head')]
    opt = torch.optim.SGD(trunk_params, lr=lr, momentum=0.9,
                          nesterov=True, weight_decay=wd)
    opt_adv = torch.optim.Adam(model.adv_head.parameters(), lr=adv_lr)

    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    crit_adv = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        if 'opt_adv' in ck:
            opt_adv.load_state_dict(ck['opt_adv'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    # FIX: `adv` is assigned inside the epoch loop. If the run is already
    # complete the loop never executes, so seed it from the history.
    adv = hist[-1]['adv'] if hist else float('nan')

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] lambda=%g  train=%d  test=%d  adv-eval=%d'
          % (tag, lamb, len(tr_idx), len(te_idx), len(adv_idx)))

    for ep in range(start, epochs):
        cur_lr = set_lr(ep)
        cur_lam = lamb * min(1.0, (ep + 1) / max(1, ramp))
        model.train(); t0 = time.time(); la = li = 0.0

        for xb, ya, yi in tl:
            xb = xb.to(DEVICE, non_blocking=True)
            ya = ya.to(DEVICE, non_blocking=True)
            yi = yi.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            opt_adv.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                oa, oi = model(xb, cur_lam)
                loss_a = crit(oa, ya)
                loss_i = crit_adv(oi, yi)
                loss = loss_a + cur_lam * loss_i
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.step(opt_adv)
            scaler.update()
            la += loss_a.item(); li += loss_i.item()

        model.eval(); corr = tot = 0
        with torch.no_grad():
            for xb, ya, yi in vl:
                xb, ya = xb.to(DEVICE), ya.to(DEVICE)
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    oa, _ = model(xb, 0.0)
                corr += (oa.argmax(1) == ya).sum().item(); tot += ya.numel()
            ac = at = 0
            for xb, ya, yi in avl:
                xb, yi = xb.to(DEVICE), yi.to(DEVICE)
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    _, oi = model(xb, 0.0)
                ac += (oi.argmax(1) == yi).sum().item(); at += yi.numel()

        acc = 100.0 * corr / tot
        adv = 100.0 * ac / max(1, at)
        hist.append({'epoch': ep, 'acc': acc, 'adv': adv,
                     'loss_a': la / len(tl), 'loss_i': li / len(tl)})
        state = {'model': model.state_dict(), 'opt': opt.state_dict(),
                 'opt_adv': opt_adv.state_dict(),
                 'epoch': ep, 'best': max(best, acc), 'hist': hist}
        torch.save(state, last)
        if acc > best:
            best = acc
            torch.save(state, best_p)
        print('  ep %2d/%d  act %.2f%%  adv %.2f%%  loss_i %.3f  lam %.2f  (%ds)'
              % (ep + 1, epochs, acc, adv, li / len(tl), cur_lam, time.time() - t0))

    json.dump(hist, open(os.path.join(P2_DIR, tag + '_history.json'), 'w'))
    print('[%s] BEST action %.2f%%  final adv %.2f%%' % (tag, best, adv))
    return {'tag': tag, 'lambda': lamb, 'action_acc': best,
            'adv_final': adv, 'ckpt': best_p}


# limited-budget attacker split: k labelled sequences per identity
def make_budget_split(k, base='reid_rand', seed=0):
    tr_idx, te_idx = SPLITS[base]
    y = np.asarray(SUBJ_LABEL)
    rng = np.random.RandomState(seed)
    keep = []
    for c in np.unique(y[tr_idx]):
        pool = np.asarray([i for i in tr_idx if y[i] == c])
        keep.extend(rng.choice(pool, size=min(k, len(pool)), replace=False))
    return np.sort(np.asarray(keep)), te_idx


for _k in [5]:
    SPLITS['reid_k%d' % _k] = make_budget_split(_k)
    print('reid_k%d: %d train / %d test'
          % (_k, len(SPLITS['reid_k%d' % _k][0]), len(SPLITS['reid_k%d' % _k][1])))

## 3 · Lambda sweep — trains only what is missing

Completed lambdas are skipped. A partially trained lambda resumes from its last
epoch. Nothing finished is repeated.

In [ ]:
LAMBDAS = [0.0, 0.1, 0.5, 1.0, 5.0, 10.0]

for lam in LAMBDAS:
    key = '%g' % lam
    r = P2.get(key)
    if r and 'action_acc' in r and os.path.exists(r.get('ckpt', '')):
        print('skipping lambda=%-5s (done, act %.2f%%)' % (key, r['action_acc']))
        continue
    P2[key] = train_defended(lam, epochs=60, lr=0.1, bs=64, workers=4)
    json.dump(P2, open(SUMMARY, 'w'), indent=2)
print('\nsweep complete')

## 4 · Shuffled-label control — own checkpoint tag

Identity labels are randomly permuted; everything else is identical to
lambda = 1. If leakage still falls, the effect came from generic regularisation
rather than identity pressure. The explicit `tag` keeps this run out of
`P2_lam1_*.pt`.

In [ ]:
if 'shuffled' in P2 and os.path.exists(P2['shuffled'].get('ckpt', '')) \
        and 'shuf' in P2['shuffled'].get('ckpt', ''):
    print('shuffled control already done:', P2['shuffled']['ckpt'])
else:
    rng_s = np.random.RandomState(7)
    ID_REAL = ID_LABEL_XSUB.copy()
    shuf = ID_LABEL_XSUB.copy()
    m = shuf >= 0
    shuf[m] = rng_s.permutation(shuf[m])
    ID_LABEL_XSUB = shuf
    try:
        P2['shuffled'] = train_defended(1.0, epochs=60, lr=0.1, bs=64,
                                        workers=4, tag='P2_shuffled')
    finally:
        ID_LABEL_XSUB = ID_REAL          # always restore
    json.dump(P2, open(SUMMARY, 'w'), indent=2)
    print('control done; identity labels restored')

## 5 · Attack every defended model — this fills Table 3

For each lambda: the convex linear probe (R2), the non-linear MLP probe (R3),
and leakage split by Group P (identities the training-time adversary saw) and
Group Q (identities it never saw). For the lambdas in `FT_LAMBDAS`: the
fine-tuning attacker (R4) at full budget and at k = 5 labelled sequences per
identity, plus a from-scratch reference at k = 5.

Probes are CPU-bound and take minutes. Each fine-tuning run is roughly 25–45
minutes, so keep `FT_LAMBDAS` short. Results are saved after every step, so an
interruption costs only the step in progress.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

FT_LAMBDAS = ['0', '1', '10']       # add '5' if time allows
SEEN_P = set(s - 1 for s in XSUB_TRAIN)


@torch.no_grad()
def defended_features(ckpt, idx, bs=256):
    m = DefendedSTGCN(60, N_ID_XSUB).to(DEVICE)
    m.load_state_dict(_load_ckpt(ckpt)['model']); m.eval()
    ld = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, idx, False), bs,
        shuffle=False, num_workers=4)
    out = []
    for xb, _, _ in ld:
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            out.append(m.features(xb.to(DEVICE)).float().cpu().numpy())
    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return np.concatenate(out, 0)


def attack_probes(ckpt):
    tr, te = SPLITS['reid_rand']
    tr, te = np.sort(tr), np.sort(te)
    Ftr, Fte = defended_features(ckpt, tr), defended_features(ckpt, te)
    sc = StandardScaler().fit(Ftr)
    Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)
    ytr, yte = np.asarray(SUBJ_LABEL)[tr], np.asarray(SUBJ_LABEL)[te]

    r = {}
    r['linear'] = 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte)
    r['mlp'] = 100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                   early_stopping=True, n_iter_no_change=15,
                                   random_state=1).fit(Ztr, ytr).score(Zte, yte)
    for nm, fn in [('P', lambda y: np.isin(y, list(SEEN_P))),
                   ('Q', lambda y: ~np.isin(y, list(SEEN_P)))]:
        mtr, mte = fn(ytr), fn(yte)
        clf = LogisticRegression(max_iter=3000).fit(Ztr[mtr], ytr[mtr])
        r['linear_' + nm] = 100 * clf.score(Zte[mte], yte[mte])
    return r


# limited-budget attacker split: k labelled sequences per identity
if 'reid_k5' not in SPLITS:
    _y = np.asarray(SUBJ_LABEL)
    _tr, _te = SPLITS['reid_rand']
    _rng = np.random.RandomState(0)
    keep = []
    for c in np.unique(_y[_tr]):
        pool = np.asarray([i for i in _tr if _y[i] == c])
        keep.extend(_rng.choice(pool, size=min(5, len(pool)), replace=False))
    SPLITS['reid_k5'] = (np.sort(np.asarray(keep)), _te)
    print('reid_k5: %d train / %d test' % (len(SPLITS['reid_k5'][0]), len(_te)))

for key in sorted(P2, key=lambda s: (not s[0].isdigit(), float(s) if s[0].isdigit() else 0)):
    r = P2[key]
    if not os.path.exists(r.get('ckpt', '')):
        print('lam=%-9s checkpoint missing, skipped' % key); continue

    if 'linear' not in r:
        t0 = time.time()
        r.update(attack_probes(r['ckpt']))
        print('lam=%-9s act %6.2f%% | R2 %6.2f%% | R3 %6.2f%% | P %6.2f%% | Q %6.2f%%  (%ds)'
              % (key, r['action_acc'], r['linear'], r['mlp'],
                 r['linear_P'], r['linear_Q'], time.time() - t0))
        json.dump(P2, open(SUMMARY, 'w'), indent=2)

    if key in FT_LAMBDAS and 'ft_full' not in r:
        trunk = export_trunk(r['ckpt'],
                             os.path.join(CKPT_DIR, 'P2_trunk_lam%s.pt' % key))
        r['ft_full'] = run_experiment(
            tag='P2_ft_lam%s' % key, labels=SUBJ_LABEL, split='reid_rand',
            num_classes=40, epochs=80, lr=0.05, bs=64,
            init_from=trunk, freeze=False)['best_acc']
        r['ft_k5'] = run_experiment(
            tag='P2_ftk5_lam%s' % key, labels=SUBJ_LABEL, split='reid_k5',
            num_classes=40, epochs=80, lr=0.05, bs=64,
            init_from=trunk, freeze=False)['best_acc']
        print('lam=%-9s R4 full %.2f%%  R4 k=5 %.2f%%' % (key, r['ft_full'], r['ft_k5']))
        json.dump(P2, open(SUMMARY, 'w'), indent=2)

# from-scratch reference at the same restricted budget (no defence involved)
if 'R5_k5' not in P2:
    P2['R5_k5'] = {'tag': 'R5_scratch_k5',
                   'action_acc': float('nan'), 'adv_final': float('nan'),
                   'ft_k5': run_experiment(
                       tag='R5_scratch_k5', labels=SUBJ_LABEL, split='reid_k5',
                       num_classes=40, epochs=80, lr=0.1, bs=64)['best_acc'],
                   'ckpt': ''}
    json.dump(P2, open(SUMMARY, 'w'), indent=2)
    print('from-scratch reference at k=5: %.2f%%' % P2['R5_k5']['ft_k5'])

## 6 · Table 3 and the privacy–utility figure

Writes `table3_tradeoff.txt`, `p2_summary.json` and `tradeoff.png`. Send me all
three. The lambda = 10 row is annotated because that run did not converge.

In [ ]:
import matplotlib.pyplot as plt

lam_keys = sorted([k for k in P2 if k[0].isdigit()], key=float)
F = lambda v: ('%.2f%%' % v) if isinstance(v, (int, float)) and v == v else '  -  '

lines = ['Table 3. Privacy-utility trade-off, NTU RGB+D 60',
         '(action = X-Sub top-1, chance 1.67%; internal adversary = 20-way head,',
         ' chance 5.00%; R2/R3/R4 = re-ID top-1 over 40 identities, chance 2.50%)',
         '',
         '%-7s %9s %9s %9s %9s %9s %9s %9s %9s' %
         ('lambda', 'action', 'adv', 'R2 lin', 'R3 MLP', 'grp P', 'grp Q',
          'R4 full', 'R4 k=5'),
         '-' * 92]
for k in lam_keys:
    r = P2[k]
    lines.append('%-7s %9s %9s %9s %9s %9s %9s %9s %9s' % (
        k, F(r.get('action_acc')), F(r.get('adv_final')), F(r.get('linear')),
        F(r.get('mlp')), F(r.get('linear_P')), F(r.get('linear_Q')),
        F(r.get('ft_full')), F(r.get('ft_k5'))))

if 'shuffled' in P2:
    r = P2['shuffled']
    lines += ['', '%-7s %9s %9s %9s %9s %9s %9s %9s %9s' % (
        'shuf', F(r.get('action_acc')), F(r.get('adv_final')), F(r.get('linear')),
        F(r.get('mlp')), F(r.get('linear_P')), F(r.get('linear_Q')),
        F(r.get('ft_full')), F(r.get('ft_k5'))),
        '  (shuffled-label control at lambda = 1)']

lines += ['',
          'Reference, no defence: re-ID from scratch = 78.86%% (full budget)']
if 'R5_k5' in P2:
    lines.append('Reference, no defence: re-ID from scratch at k=5 = %.2f%%'
                 % P2['R5_k5']['ft_k5'])
lines.append('Note: the lambda = 10 run did not converge (see history).')

txt = '\n'.join(lines)
print(txt)
open(os.path.join(P2_DIR, 'table3_tradeoff.txt'), 'w').write(txt)
json.dump(P2, open(SUMMARY, 'w'), indent=2)

xs = np.arange(len(lam_keys))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

ax[0].plot(xs, [P2[k].get('action_acc') for k in lam_keys], 'o-',
           lw=2, label='action (utility)')
for fld, lab, mk in [('adv_final', 'internal adversary', 'd'),
                     ('linear', 'R2 linear probe', 's'),
                     ('mlp', 'R3 MLP probe', '^'),
                     ('ft_full', 'R4 fine-tune (full)', 'v'),
                     ('ft_k5', 'R4 fine-tune (k=5)', 'P')]:
    px = [i for i, k in enumerate(lam_keys) if P2[k].get(fld) is not None]
    py = [P2[lam_keys[i]][fld] for i in px]
    if px:
        ax[0].plot(px, py, mk + '--', lw=1.5, label=lab)
ax[0].axhline(2.5, ls=':', c='r')
ax[0].set_xticks(xs); ax[0].set_xticklabels(lam_keys)
ax[0].set_xlabel('lambda'); ax[0].set_ylabel('accuracy (%)')
ax[0].set_title('effect of defence strength')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.25)

for fld, lab, mk in [('linear', 'R2 linear probe', 'o'),
                     ('mlp', 'R3 MLP probe', 's'),
                     ('ft_full', 'R4 fine-tune (full)', '^'),
                     ('ft_k5', 'R4 fine-tune (k=5)', 'v')]:
    px = [P2[k]['action_acc'] for k in lam_keys if P2[k].get(fld) is not None]
    py = [P2[k][fld] for k in lam_keys if P2[k].get(fld) is not None]
    if px:
        ax[1].plot(px, py, mk + '-', label=lab)
ax[1].axhline(2.5, ls=':', c='r', label='chance')
ax[1].axhline(78.86, ls='--', c='grey', label='from-scratch ceiling')
ax[1].set_xlabel('action accuracy (%)  ->  utility')
ax[1].set_ylabel('re-ID accuracy (%)  ->  leakage')
ax[1].set_title('privacy-utility trade-off')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.25)

plt.tight_layout()
plt.savefig(os.path.join(P2_DIR, 'tradeoff.png'), dpi=200)
plt.show()
print('\nwritten to', P2_DIR)

## 7 · Optional Phase 1 housekeeping

7a gives the cross-camera experiment a clean 80-epoch schedule (the saved run
resumed an old 40-epoch checkpoint, so its learning rate jumps at epoch 40).
7b trims that warm-restart prefix from the history files so the summary figure
has no discontinuity. Neither changes a reported conclusion.

In [ ]:
# 7a. clean cross-camera run (~45 min)
import glob
for f in glob.glob(os.path.join(CKPT_DIR, 'B_reid_xview_clean_*')):
    os.remove(f)

RESULTS['B_reid_xview_clean'] = run_experiment(
    tag='B_reid_xview_clean', labels=SUBJ_LABEL, split='reid_xview',
    num_classes=40, epochs=80, lr=0.1, bs=64)
save_results()
print('\nwarm-restart value was 56.26%; clean value is %.2f%%'
      % RESULTS['B_reid_xview_clean']['best_acc'])

In [ ]:
# 7b. trim the warm-restart prefix from the old histories, then regenerate the figure
import json, os
for tag in ['B_reid_rand', 'B_reid_xview']:
    p = os.path.join(RES_DIR, tag + '_history.json')
    if not os.path.exists(p):
        continue
    h = json.load(open(p))
    if len(h) > 40 and h[40]['lr'] > h[39]['lr']:
        json.dump(h[40:], open(p, 'w'))
        print('%s trimmed %d -> %d entries' % (tag, len(h), len(h[40:])))
    else:
        print('%s already clean (%d entries)' % (tag, len(h)))

## Send back

- `results/phase2/table3_tradeoff.txt`
- `results/phase2/tradeoff.png`
- `results/phase2/p2_summary.json`